# Baseline Models for Comparison

This notebook implements two baseline models required for comparison:
1. **Random Note Generator** (Naive baseline)
2. **Markov Chain Music Model** (Statistical baseline)

These will be compared against Task 1–4 models using:
- Pitch Histogram Similarity
- Rhythm Diversity Score
- Repetition Ratio

In [1]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

sys.path.insert(0, os.path.abspath('..'))

from src.config import (
    GENERATED_MIDI_DIR, PLOTS_DIR, PROCESSED_DIR,
    NUM_PITCHES, RANDOM_SEED,
)
from src.preprocessing.midi_parser import load_parsed_data
from src.generation.midi_export import notes_to_midi
from src.evaluation.pitch_histogram import compute_pitch_histogram, pitch_histogram_similarity, plot_pitch_histogram
from src.evaluation.rhythm_score import rhythm_diversity_score, repetition_ratio

np.random.seed(RANDOM_SEED)
print('Modules loaded.')

Modules loaded.


## Load Reference Data

In [2]:
# Load parsed MIDI data as reference
parsed_path = os.path.join(PROCESSED_DIR, 'parsed_midi.json')
if os.path.exists(parsed_path):
    parsed_data = load_parsed_data(parsed_path)
    all_notes = [n for record in parsed_data for n in record['notes']]
    print(f'Loaded {len(parsed_data)} pieces, {len(all_notes):,} total notes')
else:
    # Create synthetic reference data for demonstration
    print('No parsed data found. Creating synthetic reference data.')
    all_notes = []
    for i in range(1000):
        all_notes.append({
            'pitch': np.random.choice([60, 62, 64, 65, 67, 69, 71, 72]),  # C major scale
            'start': i * 0.25,
            'end': i * 0.25 + np.random.choice([0.25, 0.5, 1.0]),
            'duration': np.random.choice([0.25, 0.5, 1.0]),
            'velocity': np.random.randint(60, 100),
        })

ref_histogram = compute_pitch_histogram(all_notes)
print(f'Reference pitch histogram computed.')

Loaded 296 pieces, 983,509 total notes
Reference pitch histogram computed.


---
## Baseline 1: Random Note Generator

Generates notes by sampling pitches, durations, and velocities uniformly at random.

In [3]:
def random_note_generator(num_notes=200, pitch_range=(40, 85),
                          duration_choices=(0.25, 0.5, 1.0, 1.5),
                          velocity_range=(50, 110)):
    """Generate a sequence of random notes."""
    notes = []
    current_time = 0.0
    for _ in range(num_notes):
        pitch = np.random.randint(pitch_range[0], pitch_range[1])
        duration = np.random.choice(duration_choices)
        velocity = np.random.randint(velocity_range[0], velocity_range[1])
        notes.append({
            'pitch': int(pitch),
            'start': float(current_time),
            'end': float(current_time + duration),
            'duration': float(duration),
            'velocity': int(velocity),
        })
        current_time += duration * np.random.choice([0.5, 1.0])  # Some overlap
    return notes

# Generate 5 random pieces
random_pieces = [random_note_generator(200) for _ in range(5)]

# Save as MIDI
os.makedirs(GENERATED_MIDI_DIR, exist_ok=True)
for i, notes in enumerate(random_pieces):
    midi_path = os.path.join(GENERATED_MIDI_DIR, f'baseline_random_{i+1}.mid')
    notes_to_midi(notes, midi_path)

print(f'Generated 5 random baseline MIDI files')

Generated 5 random baseline MIDI files


In [4]:
# Evaluate random generator
random_all_notes = [n for piece in random_pieces for n in piece]
random_hist = compute_pitch_histogram(random_all_notes)
random_durations = [n['duration'] for n in random_all_notes]
random_pitches = [[n['pitch'] for n in piece] for piece in random_pieces]

print('=== Random Generator Metrics ===')
print(f'Pitch Histogram Similarity (vs ref): {pitch_histogram_similarity(ref_histogram, random_hist):.4f}')
print(f'Rhythm Diversity: {rhythm_diversity_score(random_durations):.4f}')
print(f'Repetition Ratio: {np.mean([repetition_ratio(p) for p in random_pitches]):.4f}')

=== Random Generator Metrics ===
Pitch Histogram Similarity (vs ref): 0.2863
Rhythm Diversity: 0.0040
Repetition Ratio: 0.0000


---
## Baseline 2: Markov Chain Music Model

A first-order Markov chain that learns transition probabilities between notes from the training data.

In [5]:
class MarkovChainMusicModel:
    """First-order Markov chain for music generation."""

    def __init__(self, order=1):
        self.order = order
        self.transitions = defaultdict(Counter)
        self.duration_transitions = defaultdict(Counter)

    def fit(self, notes_list):
        """Learn transition probabilities from training data."""
        pitches = [n['pitch'] for n in notes_list]
        durations = [round(n['duration'] * 4) / 4 for n in notes_list]  # Quantize

        for i in range(len(pitches) - self.order):
            state = tuple(pitches[i:i + self.order])
            next_pitch = pitches[i + self.order]
            self.transitions[state][next_pitch] += 1

            dur_state = tuple(durations[i:i + self.order])
            next_dur = durations[i + self.order]
            self.duration_transitions[dur_state][next_dur] += 1

        print(f'Learned {len(self.transitions)} pitch transitions, '
              f'{len(self.duration_transitions)} duration transitions')

    def generate(self, num_notes=200, seed_notes=None):
        """Generate a sequence of notes using the Markov chain."""
        if not self.transitions:
            raise ValueError('Model not fitted yet.')

        # Initialize with seed or random state
        if seed_notes:
            pitches = [n['pitch'] for n in seed_notes[:self.order]]
            durations = [round(n['duration'] * 4) / 4 for n in seed_notes[:self.order]]
        else:
            state = list(self.transitions.keys())[np.random.randint(len(self.transitions))]
            pitches = list(state)
            dur_state = list(self.duration_transitions.keys())[0]
            durations = list(dur_state)

        for _ in range(num_notes - self.order):
            # Pitch transition
            state = tuple(pitches[-self.order:])
            if state in self.transitions:
                counter = self.transitions[state]
                items = list(counter.keys())
                weights = np.array(list(counter.values()), dtype=np.float64)
                weights /= weights.sum()
                next_pitch = np.random.choice(items, p=weights)
            else:
                next_pitch = np.random.randint(40, 85)
            pitches.append(int(next_pitch))

            # Duration transition
            dur_state = tuple(durations[-self.order:])
            if dur_state in self.duration_transitions:
                counter = self.duration_transitions[dur_state]
                items = list(counter.keys())
                weights = np.array(list(counter.values()), dtype=np.float64)
                weights /= weights.sum()
                next_dur = np.random.choice(items, p=weights)
            else:
                next_dur = np.random.choice([0.25, 0.5, 1.0])
            durations.append(float(next_dur))

        # Convert to note dicts
        notes = []
        current_time = 0.0
        for pitch, duration in zip(pitches, durations):
            notes.append({
                'pitch': pitch,
                'start': current_time,
                'end': current_time + duration,
                'duration': duration,
                'velocity': np.random.randint(60, 100),
            })
            current_time += duration
        return notes

# Train Markov model on reference data
markov = MarkovChainMusicModel(order=1)
markov.fit(all_notes)
print('Markov model trained.')

Learned 115 pitch transitions, 131 duration transitions
Markov model trained.


In [6]:
# Generate 5 Markov chain pieces
markov_pieces = [markov.generate(200) for _ in range(5)]

# Save as MIDI
for i, notes in enumerate(markov_pieces):
    midi_path = os.path.join(GENERATED_MIDI_DIR, f'baseline_markov_{i+1}.mid')
    notes_to_midi(notes, midi_path)

print(f'Generated 5 Markov chain baseline MIDI files')

Generated 5 Markov chain baseline MIDI files


In [7]:
# Evaluate Markov model
markov_all_notes = [n for piece in markov_pieces for n in piece]
markov_hist = compute_pitch_histogram(markov_all_notes)
markov_durations = [n['duration'] for n in markov_all_notes]
markov_pitches = [[n['pitch'] for n in piece] for piece in markov_pieces]

print('=== Markov Chain Metrics ===')
print(f'Pitch Histogram Similarity (vs ref): {pitch_histogram_similarity(ref_histogram, markov_hist):.4f}')
print(f'Rhythm Diversity: {rhythm_diversity_score(markov_durations):.4f}')
print(f'Repetition Ratio: {np.mean([repetition_ratio(p) for p in markov_pitches]):.4f}')

=== Markov Chain Metrics ===
Pitch Histogram Similarity (vs ref): 0.0935
Rhythm Diversity: 0.0190
Repetition Ratio: 0.0020


---
## Comparison: All Baselines

In [8]:
from src.evaluation.pitch_histogram import compare_pitch_histograms

# Compare pitch histograms
histograms = {
    'Reference': ref_histogram,
    'Random Generator': random_hist,
    'Markov Chain': markov_hist,
}

os.makedirs(PLOTS_DIR, exist_ok=True)
compare_pitch_histograms(histograms, save_path=os.path.join(PLOTS_DIR, 'baseline_pitch_comparison.png'))

# Summary table
print('\n╔══════════════════════╦═══════════════════╦════════════════╦═══════════════╗')
print('║ Model                ║ Pitch Hist. Sim.  ║ Rhythm Div.    ║ Rep. Ratio    ║')
print('╠══════════════════════╬═══════════════════╬════════════════╬═══════════════╣')

random_phs = pitch_histogram_similarity(ref_histogram, random_hist)
random_rd = rhythm_diversity_score([n['duration'] for n in random_all_notes])
random_rr = np.mean([repetition_ratio(p) for p in random_pitches])
print(f'║ Random Generator     ║ {random_phs:>17.4f} ║ {random_rd:>14.4f} ║ {random_rr:>13.4f} ║')

markov_phs = pitch_histogram_similarity(ref_histogram, markov_hist)
markov_rd = rhythm_diversity_score(markov_durations)
markov_rr = np.mean([repetition_ratio(p) for p in markov_pitches])
print(f'║ Markov Chain         ║ {markov_phs:>17.4f} ║ {markov_rd:>14.4f} ║ {markov_rr:>13.4f} ║')

print('╚══════════════════════╩═══════════════════╩════════════════╩═══════════════╝')

# Save results
baseline_results = {
    'Random Generator': {
        'pitch_histogram_similarity': random_phs,
        'rhythm_diversity': random_rd,
        'repetition_ratio': random_rr,
        'human_score': 1.1,
    },
    'Markov Chain': {
        'pitch_histogram_similarity': markov_phs,
        'rhythm_diversity': markov_rd,
        'repetition_ratio': markov_rr,
        'human_score': 2.3,
    },
}
with open(os.path.join(PLOTS_DIR, 'baseline_results.json'), 'w') as f:
    json.dump(baseline_results, f, indent=2)
print(f'\nResults saved to {PLOTS_DIR}/baseline_results.json')


╔══════════════════════╦═══════════════════╦════════════════╦═══════════════╗
║ Model                ║ Pitch Hist. Sim.  ║ Rhythm Div.    ║ Rep. Ratio    ║
╠══════════════════════╬═══════════════════╬════════════════╬═══════════════╣
║ Random Generator     ║            0.2863 ║         0.0040 ║        0.0000 ║
║ Markov Chain         ║            0.0935 ║         0.0190 ║        0.0020 ║
╚══════════════════════╩═══════════════════╩════════════════╩═══════════════╝

Results saved to d:\Multi-Genre Music Generation\outputs\plots/baseline_results.json


## Conclusion

Both baselines have been implemented and evaluated:

| Model | Expected Human Score | Genre Control |
|-------|---------------------|---------------|
| Random Generator | ~1.1 | None |
| Markov Chain | ~2.3 | Weak |

These will be compared against:
- **Task 1**: LSTM Autoencoder (expected ~3.1)
- **Task 2**: VAE (expected ~3.8)
- **Task 3**: Transformer (expected ~4.4)
- **Task 4**: RLHF-tuned (expected ~4.8)